<a href="https://colab.research.google.com/github/Logan-Das/MMA3001_Project/blob/main/notebooks/02_grouped_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pork rasher QC: full pipeline (Google Colab)

Runs every experiment in the report in order. Set **Runtime → Change runtime type → T4 GPU** first.
Store your Roboflow key in Colab **Secrets** as `ROBOFLOW_API_KEY` (never paste it into the notebook).
All outputs go to Google Drive so a disconnect does not lose them.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/pork_rasher"

# Clone the repo (uses the GIT_TOKEN secret if the repo is private)
try:
    token = userdata.get("GIT_TOKEN")
    REPO = f"https://{token}@github.com/Logan-Das/MMA3001_Project.git"
except Exception:
    REPO = "https://github.com/Logan-Das/MMA3001_Project.git"
!git clone -q {REPO} /content/MMA3001_Project
%cd /content/MMA3001_Project
!pip install -q -r requirements.txt && pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Make the package importable without restarting the runtime
import sys
sys.path.insert(0, "/content/MMA3001_Project/src")

## 1. Download the dataset (Version 4)
Every public version is augmented and the dataset cannot be copied into a free workspace (1,000-image limit), so Version 4 is used as is. Its training images include colour-augmented copies of each frame; the split script keeps one copy per frame (the un-augmented valid/test copy where one exists) before re-splitting.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
VERSION = 4   # augmented copies are removed in step 2
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
ds = rf.workspace("hello-2aqe0").project("pork-rasher-error-packaging").version(VERSION).download("yolov11")
SRC = ds.location
print(SRC)

## 2. Leakage check and grouped split
Dry run first: check how many augmented duplicates were removed, then read the gap percentiles and the number of groups the ORIGINAL split shares between splits.
Choose `--max-gap` where the gaps jump from "same tray" to "new tray", then build the split.

In [ ]:
!python scripts/make_grouped_split.py --src "$SRC" --max-gap 15 --dry-run

In [ ]:
MAX_GAP = 15   # <- set from the data checks (notebook 03); justify it in the report
!python scripts/make_grouped_split.py --src "$SRC" --dst /content/pork_grouped --max-gap $MAX_GAP --seed 0
!cp /content/pork_grouped/split_manifest.csv $OUT/

## 3. Train the baseline and the alternative with identical settings

In [ ]:
!python scripts/train.py --model yolo11n.pt --data /content/pork_grouped/data.yaml --out $OUT --name yolo11n_grouped

In [ ]:
!python scripts/train.py --model yolo11s.pt --data /content/pork_grouped/data.yaml --out $OUT --name yolo11s_grouped

## 4. Detection metrics per class (test split)

In [ ]:
from ultralytics import YOLO
for run in ["yolo11n_grouped", "yolo11s_grouped"]:
    m = YOLO(f"{OUT}/{run}/weights/best.pt").val(data="/content/pork_grouped/data.yaml", split="test",
                                                 project=OUT, name=f"{run}_test")

## 5. Tray-level pass/fail (threshold chosen on validation)

In [ ]:
for run in ["yolo11n_grouped", "yolo11s_grouped"]:
    !python scripts/evaluate_trays.py --weights $OUT/{run}/weights/best.pt --data /content/pork_grouped --out $OUT/results/{run}

In [ ]:
# Per-tray scores and histogram (threshold from tray_summary.json)
import json
for run in ["yolo11n_grouped", "yolo11s_grouped"]:
    t = json.load(open(f"{OUT}/results/{run}/tray_summary.json"))["threshold_from_val"]
    !python scripts/plot_tray_scores.py --data /content/pork_grouped --scores $OUT/results/{run}/max_conf_test.npy --threshold {t} --out $OUT/results/{run} --prefix {run.split("_")[0]}

## 6. Latency (warm-up, repeats, FP32 vs FP16)

In [ ]:
GFLOPS = {"yolo11n_grouped": 6.4, "yolo11s_grouped": 21.5}   # <- check the 'summary (fused)' line of each run
for run, g in GFLOPS.items():
    for half in ["", "--half"]:
        !python scripts/benchmark.py --weights $OUT/{run}/weights/best.pt --images /content/pork_grouped/test/images --gflops {g} {half} --out $OUT/results/{run}

## 7. Profile one prediction
Shows where time goes (pre-processing, model, NMS). Use this to explain the benchmark numbers.

In [ ]:
import cProfile, pstats
from ultralytics import YOLO
from porkqc.labels import list_images
model = YOLO(f"{OUT}/yolo11n_grouped/weights/best.pt")
imgs = [str(p) for p in list_images("/content/pork_grouped/test/images")][:50]
model.predict(imgs[0], verbose=False)   # warm-up
cProfile.run("for p in imgs: model.predict(p, verbose=False)", "/content/pred.prof")
pstats.Stats("/content/pred.prof").sort_stats("cumulative").print_stats(15)